### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
cd Your_Dir/emg2qwerty

### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [ ]:
!pip install -r requirements.txt

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

# RUN THIS BEFORE TRAINING

In [ ]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

## Baseline

#### Training baseline:

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [ ]:
# Check the files are imported from the correct location (not from a different installation)
import emg2qwerty
import emg2qwerty.lightning as L
import emg2qwerty.data as D

print("emg2qwerty package:", emg2qwerty.__file__)
print("lightning.py:", L.__file__)
print("data.py:", D.__file__)

emg2qwerty package: /home/aki.hj/emg2qwerty/emg2qwerty/__init__.py
lightning.py: /home/aki.hj/emg2qwerty/emg2qwerty/lightning.py
data.py: /home/aki.hj/emg2qwerty/emg2qwerty/data.py


In [13]:
# Single-user training
!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1 \
  trainer.max_epochs=40 \
  # --multirun


[2026-03-11 07:10:12,698][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

#### Testing baseline:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun

## CNN + BiGRU + CTC

#### Training CNN + BiGRU + CTC:

In [8]:
!HYDRA_FULL_ERROR=1 python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.max_epochs=40 \
  module._target_=emg2qwerty.lightning.TDSRNNCTCModule \
  +module.rnn_type=gru \
  +module.rnn_hidden=256 \
  +module.rnn_layers=2 \
  +module.bidirectional=true \
  +module.rnn_dropout=0.1

[2026-03-12 22:06:24,582][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

#### Testing CNN + BiGRU + CTC:

In [5]:
!HYDRA_FULL_ERROR=1 python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="/home/aki.hj/emg2qwerty/logs/2026-03-12/20-40-11/checkpoints/last.ckpt" \
  train=False \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  module._target_=emg2qwerty.lightning.TDSRNNCTCModule \
  +module.rnn_hidden=256 \
  +module.rnn_layers=2 \
  +module.bidirectional=true \
  +module.rnn_dropout=0.1

[2026-03-12 22:01:08,950][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

### Hyperparameter Sweep

In [ ]:
import os
import subprocess
from pathlib import Path
import pandas as pd
import ast

# Small first sweep: 8 runs
learning_rates = [1e-3, 5e-4, 3e-4, 1e-4]
batch_sizes = [16, 32]

# Keep the best architecture fixed
kernel_width = 32
block_channels = "[24,24,24,24]"

results = []
log_dir = Path("sweep_logs_stage1")
log_dir.mkdir(exist_ok=True)

def parse_log(logfile):
    text = Path(logfile).read_text(errors="ignore")

    # Find the final printed Python dict
    start = text.rfind("{'val_metrics':")
    if start == -1:
        raise ValueError(f"Could not find final metrics dict in {logfile}")

    metrics = ast.literal_eval(text[start:])

    val_cer = metrics["val_metrics"][0]["val/CER"]
    test_cer = metrics["test_metrics"][0]["test/CER"]
    best_ckpt = metrics["best_checkpoint"]

    return val_cer, test_cer, best_ckpt

for lr in learning_rates:
    for bs in batch_sizes:
        run_name = f"lr{lr}_bs{bs}_kw{kernel_width}"
        log_file = log_dir / f"{run_name}.txt"

        cmd = f"""
        TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python -m emg2qwerty.train \
          user="single_user" \
          trainer.accelerator=gpu \
          trainer.devices=1 \
          trainer.max_epochs=40 \
          datamodule.batch_size={bs} \
          module.kernel_width={kernel_width} \
          module.block_channels='{block_channels}' \
          optimizer.lr={lr} \
          2>&1 | tee {log_file}
        """

        print("\n" + "="*100)
        print("RUNNING:", run_name)
        print("="*100)

        ret = subprocess.run(["bash", "-lc", cmd], text=True)

        try:
            val_cer, test_cer, best_ckpt = parse_log(log_file)
        except Exception as e:
            print(f"Failed to parse {log_file}: {e}")
            val_cer, test_cer, best_ckpt = None, None, None

        results.append({
            "run_name": run_name,
            "lr": lr,
            "batch_size": bs,
            "kernel_width": kernel_width,
            "block_channels": block_channels,
            "val_CER": val_cer,
            "test_CER": test_cer,
            "best_checkpoint": best_ckpt,
            "return_code": ret.returncode,
            "log_file": str(log_file),
        })


In [ ]:
df_stage1 = pd.DataFrame(results).sort_values("val_CER", ascending=True, na_position="last")
df_stage1.to_csv("sweep_stage1_results.csv", index=False)
df_stage1

# Ablation Experiments

### Investigate the relationship between the number of electrode channels and CER

Training:

In [10]:
%%bash

python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.max_epochs=40 \
  channel_mask.mode=all \
  channel_mask.keep_total_channels=32 \
  module._target_=emg2qwerty.lightning.TDSRNNCTCModule \
  +module.rnn_hidden=256 \
  +module.rnn_layers=2 \
  +module.bidirectional=true \
  +module.rnn_dropout=0.1 \
  2>&1 | tee run_32_all.txt


for keep in 16 8 4; do \
python -m emg2qwerty.train \
    user="single_user" \
    trainer.accelerator=gpu \
    trainer.devices=1 \
    trainer.max_epochs=40 \
    channel_mask.mode=random_subset \
    channel_mask.keep_total_channels=$keep \
    channel_mask.seed=1501 \
    module._target_=emg2qwerty.lightning.TDSRNNCTCModule \
    +module.rnn_hidden=256 \
    +module.rnn_layers=2 \
    +module.bidirectional=true \
    +module.rnn_dropout=0.1 \
    2>&1 | tee run_${keep}_random.txt
done

python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.max_epochs=40 \
  channel_mask.mode=left_only \
  module._target_=emg2qwerty.lightning.TDSRNNCTCModule \
  +module.rnn_hidden=256 \
  +module.rnn_layers=2 \
  +module.bidirectional=true \
  +module.rnn_dropout=0.1 \
  2>&1 | tee run_left_only.txt

Error while terminating subprocess (pid=62770): 


TypeError: %d format: a real number is required, not NoneType

### Investigate the relationship between sampling rate and CER

Training:

In [ ]:
# Quick test before sampling rate experiment to make sure it runs
!python -m emg2qwerty.train user="single_user" trainer.accelerator=gpu trainer.devices=1 trainer.max_epochs=1
# downsample.factor=2

In [ ]:
%%bash

for factor in 1 2 4 8; do \
python -m emg2qwerty.train \
    user="single_user" \
    trainer.accelerator=gpu \
    trainer.devices=1 \
    trainer.max_epochs=40 \
    downsample.factor=$factor \
    module._target_=emg2qwerty.lightning.TDSRNNCTCModule \
    +module.rnn_hidden=256 \
    +module.rnn_layers=2 \
    +module.bidirectional=true \
    +module.rnn_dropout=0.1 \
    2>&1 | tee run_${factor}_downsampled.txt
done

### Investigate the relationship between the amount of training data and CER

Training:

In [ ]:
# Quick test before training data experiment to make sure it runs
!python -m emg2qwerty.train user="single_user" trainer.accelerator=gpu trainer.devices=1 trainer.max_epochs=1 ++datamodule.train_fraction=0.10 ++datamodule.subset_seed=1501

In [ ]:
%%bash

for frac in 0.10 0.25 1.0; do \
  python -m emg2qwerty.train \
    user="single_user" \
    trainer.accelerator=gpu \
    trainer.devices=1 \
    trainer.max_epochs=40 \
    ++datamodule.train_fraction=$frac \
    ++datamodule.subset_seed=1501 \
    module._target_=emg2qwerty.lightning.TDSRNNCTCModule \
    +module.rnn_hidden=256 \
    +module.rnn_layers=2 \
    +module.bidirectional=true \
    +module.rnn_dropout=0.1 \
    2>&1 | tee run_train_${frac}.txt
done